# 01_multi_provider_architectures/
In production engineering, hardcoding an application to a single LLM vendor creates severe vendor lock-in and systemic vulnerability. Enterprise applications demand provider-agnostic design patterns.

This section breaks down how the Big Three ecosystem APIs differ fundamentally under the hood, how to implement unified abstraction, and how to handle production-grade resilience.

## 1. Structural Schema Comparison
Interviewers frequently test whether you understand how provider payloads vary beyond simple Python wrappers.

| Feature / Dimension | OpenAI (openai) | Anthropic (anthropic) | Google Gemini (google-genai)
| :---|:---|:---|:---|
| System Instructions | "Passed as a message object with role: ""system"" inside the messages array." | "Passed as a top-level explicit parameter (system=""..."") separate from messages." | "Passed via config=types.GenerateContentConfig(system_instruction=""..."")."
| Message History Schema | "Strict alternating array of objects: {""role"": ""user"" / ""assistant"", ""content"": ""...""}." | "Strict alternating array: {""role"": ""user"" / ""assistant"", ""content"": ""...""}." | "Uses contents parameter, accepting strings, PIL images, or structured content parts."
| Client Initialization | client = OpenAI(api_key=...) | client = anthropic.Anthropic(api_key=...) |client = genai.Client(api_key=...)

## 2. Production Implementation Code
Here is a clean implementation showing how to query OpenAI and Anthropic using their official SDK structures

In [ ]:
import os
from openai import OpenAI
import anthropic

# ==========================================
# 1. OpenAI Client Interaction Pattern
# ==========================================
def call_openai():
    client = OpenAI(api_key=os.environ.get("OPENAI_API_KEY"))
    
    response = client.chat.completions.create(
        model="gpt-4o-mini",
        messages=[
            {"role": "system", "content": "You are a cloud infrastructure architect."},
            {"role": "user", "content": "Explain blue-green deployments in 2 sentences."}
        ],
        temperature=0.2
    )
    return response.choices[0].message.content

# ==========================================
# 2. Anthropic Client Interaction Pattern
# ==========================================
def call_anthropic():
    client = anthropic.Anthropic(api_key=os.environ.get("ANTHROPIC_API_KEY"))
    
    response = client.messages.create(
        model="claude-3-5-sonnet-20241022",
        max_tokens=300,
        # Notice: System prompt is a top-level parameter, NOT part of the messages array
        system="You are a senior security compliance officer.",
        messages=[
            {"role": "user", "content": "What is SOC 2 Type II compliance in 2 sentences?"}
        ]
    )
    return response.content[0].text

if __name__ == "__main__":
    print("--- OpenAI Output ---")
    print(call_openai())
    print("\n--- Anthropic Output ---")
    print(call_anthropic())

## 3. Deep-Dive: Architecture & Resilience
When asked about multi-provider design in an interview, touch upon these three architectural pillars:

**The Abstract Factory Pattern:** Do not scatter raw SDK calls across your codebase. Wrap providers behind a unified interface or abstract base class (ABC) so your service layer calls a generic generate_text(prompt, system_prompt) method, hiding whether OpenAI, Anthropic, or an internal local model fulfills the request.

**Transient Error Handling (429 & 5xx):** LLM APIs experience frequent rate limits (429 Too Many Requests) and gateway timeouts (502/504). Never execute straight linear requests in production. Implement Exponential Backoff with Jitter using libraries like tenacity to retry failed requests automatically without overwhelming provider endpoints.

**Context Window Degradation:** Remember that chat histories grow quadratically in stateful client wrappers. If your application handles long threads, you must implement a sliding window truncation strategy or token-budget summarization daemon to strip old messages before payloads hit the API.